# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [5]:
from pyspark.sql import functions as F

Add a column that creates a unique key to identify each record

In [6]:
#we add a unique id to each trip
df_trips = df_trips.withColumn("trip_id", F.monotonically_increasing_id())
# cache the dataframe so the ids stay the same between the queries
df_trips = df_trips.cache()

# is the key unique
print("number of trips:", df_trips.count())
print("number of distinct ids:", df_trips.select("trip_id").distinct().count())
df_trips.select("trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance").show(5)

number of trips: 7696617
number of distinct ids: 7696617
+-----------+--------------------+---------------------+-------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|
+-----------+--------------------+---------------------+-------------+
|60129542144| 2019-01-01 00:46:40|  2019-01-01 00:53:20|          1.5|
|60129542145| 2019-01-01 00:59:47|  2019-01-01 01:18:59|          2.6|
|60129542146| 2018-12-21 13:48:30|  2018-12-21 13:52:40|          0.0|
|60129542147| 2018-11-28 15:52:25|  2018-11-28 15:55:45|          0.0|
|60129542148| 2018-11-28 15:56:57|  2018-11-28 15:58:33|          0.0|
+-----------+--------------------+---------------------+-------------+
only showing top 5 rows


`monotonically_increasing_id()` gives a unique id to each row. The ids are unique but not consecutive because the partition id is stored in the upper bits.  
We cache the dataframe so the ids are computed once and stay the same for all the next questions. 

 Which trip has the highest passenger count

In [7]:
max_passengers = df_trips.agg(F.max("passenger_count")).first()[0]
print("highest passenger count:", max_passengers)

df_trips.filter(F.col("passenger_count") == max_passengers) \
    .select("trip_id", "tpep_pickup_datetime", "passenger_count", "trip_distance", "total_amount") \
    .show()

highest passenger count: 9.0
+-----------+--------------------+---------------+-------------+------------+
|    trip_id|tpep_pickup_datetime|passenger_count|trip_distance|total_amount|
+-----------+--------------------+---------------+-------------+------------+
|60130492100| 2019-01-05 13:12:29|            9.0|          0.0|        12.6|
|60130838431| 2019-01-07 03:19:36|            9.0|          0.0|         9.3|
|60131554242| 2019-01-10 00:43:10|            9.0|          0.0|        11.3|
|60132426139| 2019-01-13 04:13:24|            9.0|          0.0|       12.25|
|60134076851| 2019-01-19 16:45:25|            9.0|          0.0|      110.76|
|60134394369| 2019-01-21 03:46:51|            9.0|          0.0|       12.74|
|60134539934| 2019-01-21 19:20:28|            9.0|          0.0|         9.8|
|60136828827| 2019-01-30 18:34:12|            9.0|          0.0|        10.3|
|60136916020| 2019-01-30 22:17:51|            9.0|        13.38|        90.8|
+-----------+--------------------+-

The highest passenger count is 9, and it's not a single trip: 9 trips have 9 passengers.  
8 of them have a distance of 0, so they look more like data entry errors than real trips . The only one with a real distance is trp 60136916020 .

#### What is the Average passenger count

In [8]:
df_trips.agg(F.avg("passenger_count").alias("avg_passenger_count")).show()

+-------------------+
|avg_passenger_count|
+-------------------+
| 1.5670317144945614|
+-------------------+



On average there are about 1.57 passengers per trip, most of the trips have only 1 passenger. The rows where passenger_count is null are ignored by avg.

#### Shortest/longest trip by distance? by time?

In [9]:
#trip duration in minutes
df_trips = df_trips.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

cols = ["trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "trip_duration_min"]

print("shortest trip by distance")
df_trips.orderBy(F.asc("trip_distance")).select(cols).show(1)
print("number of trips with a distance of 0:", df_trips.filter(F.col("trip_distance") == 0).count())

print("longest trip by distance")
df_trips.orderBy(F.desc("trip_distance")).select(cols).show(1)

shortest trip by distance
+-----------+--------------------+---------------------+-------------+-----------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|trip_duration_min|
+-----------+--------------------+---------------------+-------------+-----------------+
|60129542146| 2018-12-21 13:48:30|  2018-12-21 13:52:40|          0.0|4.166666666666667|
+-----------+--------------------+---------------------+-------------+-----------------+
only showing top 1 row
number of trips with a distance of 0: 55089
longest trip by distance
+-----------+--------------------+---------------------+-------------+-----------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|trip_duration_min|
+-----------+--------------------+---------------------+-------------+-----------------+
|60135616235| 2019-01-25 21:56:39|  2019-01-25 22:06:08|        831.8|9.483333333333333|
+-----------+--------------------+---------------------+-------------+-----------

In [10]:
print("shortest trip by time")
df_trips.orderBy(F.asc("trip_duration_min")).select(cols).show(1)
print("longest trip by time")
df_trips.orderBy(F.desc("trip_duration_min")).select(cols).show(1)

shortest trip by time
+-----------+--------------------+---------------------+-------------+-----------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|trip_duration_min|
+-----------+--------------------+---------------------+-------------+-----------------+
|60130745328| 2019-01-06 15:15:08|  2018-11-09 02:34:38|          3.3|         -84280.5|
+-----------+--------------------+---------------------+-------------+-----------------+
only showing top 1 row
longest trip by time
+-----------+--------------------+---------------------+-------------+-----------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|trip_duration_min|
+-----------+--------------------+---------------------+-------------+-----------------+
|60129610411| 2019-01-01 07:01:20|  2019-01-31 14:29:21|          1.2|43648.01666666667|
+-----------+--------------------+---------------------+-------------+-----------------+
only showing top 1 row


by distance:
- Shortest: 0 miles. There is no single shortest trip, **55,089 trips** have a distance of 0 (`show(1)` only shows the first one).
- Longest: trip `60135616235` with 831.8 miles in 9.5 minutes.

By time:
- Shortest: trip `60130745328` .
- Longest: trip `60129610411` .

So the raw min/max values are all data errors, we come back to them in the outliers question.

#### busiest day/slowest single day

In [11]:
# some pickup dates are not in january 2019 (see the outliers question), so we only keep january 2019
df_jan_2019 = df_trips.filter(
    (F.col("tpep_pickup_datetime") >= "2019-01-01") & (F.col("tpep_pickup_datetime") < "2019-02-01")
)

daily_trips = df_jan_2019.groupBy(F.to_date("tpep_pickup_datetime").alias("pickup_date")).count()

print("busiest day")
daily_trips.orderBy(F.desc("count")).show(1)

print("slowest day")
daily_trips.orderBy(F.asc("count")).show(1)

busiest day
+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292499|
+-----------+------+
only showing top 1 row
slowest day
+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-01|189432|
+-----------+------+
only showing top 1 row


- Busiest day: **Friday 2019-01-25** with **292,499 trips**.
- Slowest day: **2019-01-01** (New Year's Day) with **189,432 trips**.

We only kept the pickups of January 2019, otherwise the slowest days would be wrong dates like 2001-02-02 with only 1 trip.

#### busiest/slowest time of day

In [12]:
# number of trips by pickup hour
hourly_trips = df_jan_2019.groupBy(F.hour("tpep_pickup_datetime").alias("pickup_hour")).count()

hourly_trips.orderBy(F.desc("count")).show(24)

+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|515374|
|         19|475152|
|         17|468407|
|         15|452679|
|         14|433115|
|         20|423128|
|         16|420806|
|         21|409873|
|         13|404149|
|         12|401172|
|         11|375438|
|          8|373735|
|         22|369026|
|          9|365924|
|         10|361382|
|          7|304858|
|         23|281812|
|          0|207758|
|          6|178598|
|          1|149242|
|          2|109413|
|          3| 78084|
|          5| 75532|
|          4| 61423|
+-----------+------+



- Busiest time: **18h (6pm-7pm)** with 515,374 trips. 17h-19h are the top 3, it's the evening rush hour.
- Slowest time: **4h (4am-5am)** with 61,423 trips.

The late night (1h-6h) is the slowest part of the day and the evening is the busiest. There is also a smaller peak in the morning (8h-9h).

#### On average which day of the week is slowest/busiest

In [13]:
# january 2019 has 5 tuesdays, wednesdays and thursdays but only 4 of the other days,
# so we take the average number of trips per day instead of the total
daily_trips.withColumn("day_of_week", F.date_format("pickup_date", "EEEE")) \
    .groupBy("day_of_week") \
    .agg(F.avg("count").alias("avg_trips_per_day"), F.count("*").alias("nb_days")) \
    .orderBy(F.desc("avg_trips_per_day")) \
    .show()

+-----------+-----------------+-------+
|day_of_week|avg_trips_per_day|nb_days|
+-----------+-----------------+-------+
|     Friday|         271787.5|      4|
|   Thursday|         271398.4|      5|
|  Wednesday|         253045.8|      5|
|   Saturday|        252494.75|      4|
|    Tuesday|         241815.2|      5|
|     Monday|         226941.0|      4|
|     Sunday|         214972.5|      4|
+-----------+-----------------+-------+



- Busiest: **Friday** with about 271,788 trips per day on average (Thursday is very close).
- Slowest: **Sunday** with about 214,973 trips per day, then Monday.

The Tuesday average is a bit lowered by New Year's Day (2019-01-01 was a Tuesday).

#### Does trip distance or num passangers affect tip amount

In [14]:
# tips are only recorded for credit card payments (payment_type = 1), cash tips are not in the data
df_card = df_trips.filter(F.col("payment_type") == 1)

df_card.select(
    F.corr("trip_distance", "tip_amount").alias("corr_distance_tip"),
    F.corr("passenger_count", "tip_amount").alias("corr_passengers_tip")
).show()

df_card.groupBy("passenger_count") \
    .agg(F.avg("tip_amount").alias("avg_tip"), F.count("*").alias("nb_trips")) \
    .orderBy("passenger_count") \
    .show()

+------------------+--------------------+
| corr_distance_tip| corr_passengers_tip|
+------------------+--------------------+
|0.6715943173994262|0.010141706333933796|
+------------------+--------------------+

+---------------+------------------+--------+
|passenger_count|           avg_tip|nb_trips|
+---------------+------------------+--------+
|            0.0| 2.519602931459125|   83235|
|            1.0|2.5339521622183447| 3936888|
|            2.0| 2.614486572689903|  781318|
|            3.0| 2.585738670425347|  218521|
|            4.0|2.6029785593256283|   92068|
|            5.0|2.6182445012300475|  231279|
|            6.0| 2.609179752008406|  142908|
|            7.0| 11.30090909090909|      11|
|            8.0|  6.96074074074074|      27|
|            9.0|           3.50625|       8|
+---------------+------------------+--------+



We only use credit card payments because cash tips are not included in the data (see the data dictionary).
- **Trip distance: yes.** The correlation between distance and tip is **0.67**, longer trips have higher fares and people usually tip a percentage of the fare.
- **Number of passengers: no.** The correlation is **0.01** and the average tip is almost the same (about $2.5-2.6) for 0 to 6 passengers. The values for 7 to 9 passengers are based on only 8 to 27 trips, so they don't mean much.

#### What was the highest "extra" charge and which trip

In [15]:
df_trips.orderBy(F.desc("extra")) \
    .select("trip_id", "tpep_pickup_datetime", "trip_distance", "fare_amount", "extra", "total_amount") \
    .show(1)

+-----------+--------------------+-------------+-----------+------+------------+
|    trip_id|tpep_pickup_datetime|trip_distance|fare_amount| extra|total_amount|
+-----------+--------------------+-------------+-----------+------+------------+
|60134865627| 2019-01-23 08:58:09|          0.0|  355676.98|535.38|   356214.78|
+-----------+--------------------+-------------+-----------+------+------------+
only showing top 1 row


The highest extra charge is **$535.38**, for trip `60134865627` (2019-01-23 at 08:58).  
Normally `extra` is only $0.50 or $1 (night and rush hour surcharges), and this trip has a distance of 0, a duration of 0 and a fare of $355,676.98, so it is clearly an error.

#### Are there any datapoints that seem to be strange/outliers

In [16]:
print("pickups not in january 2019:", df_trips.count() - df_jan_2019.count())
print("dropoff before pickup:", df_trips.filter(F.col("trip_duration_min") < 0).count())
print("trips longer than 24 hours:", df_trips.filter(F.col("trip_duration_min") > 24 * 60).count())
print("trips with 0 passengers:", df_trips.filter(F.col("passenger_count") == 0).count())
print("trips with a distance of 0:", df_trips.filter(F.col("trip_distance") == 0).count())
print("negative total amount:", df_trips.filter(F.col("total_amount") < 0).count())
print("RatecodeID not between 1 and 6:", df_trips.filter(~F.col("RatecodeID").between(1, 6)).count())

# pickup dates outside january 2019
df_trips.groupBy(F.year("tpep_pickup_datetime").alias("year"), F.month("tpep_pickup_datetime").alias("month")) \
    .count() \
    .orderBy("year", "month") \
    .show(50)

# highest fares
df_trips.orderBy(F.desc("fare_amount")) \
    .select("trip_id", "tpep_pickup_datetime", "trip_distance", "trip_duration_min", "fare_amount", "total_amount") \
    .show(3)

pickups not in january 2019: 537
dropoff before pickup: 4
trips longer than 24 hours: 5
trips with 0 passengers: 117381
trips with a distance of 0: 55089
negative total amount: 7127
RatecodeID not between 1 and 6: 252
+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2001|    2|      1|
|2003|    1|      2|
|2008|   12|     22|
|2009|    1|     50|
|2018|   11|     11|
|2018|   12|    355|
|2019|    1|7696080|
|2019|    2|     72|
|2019|    3|      5|
|2019|    4|      6|
|2019|    5|      1|
|2019|    6|      2|
|2019|    7|      6|
|2019|    8|      1|
|2019|    9|      1|
|2088|    1|      2|
+----+-----+-------+

+-----------+--------------------+-------------+-----------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|trip_distance|trip_duration_min|fare_amount|total_amount|
+-----------+--------------------+-------------+-----------------+-----------+------------+
|60132041799| 2019-01-11 19:33:15|          2.4|             19.9|  623259.86|   6232

Yes, there are a lot of strange values. We consider a value strange when it is physically impossible, not consistent with the other columns of the row, or not in the data dictionary:

- **Wrong dates**: 537 pickups are not in January 2019, from 2001 up to 2088 (in the future). The file should only contain January 2019.
- **Wrong durations**: 4 trips with the dropoff before the pickup, and 5 trips longer than 24 hours.
- **Wrong distances**: 831.8 miles in 9.5 minutes, and 55,089 trips with a distance of 0 (cancelled trips or meter not used properly).
- **Passengers**: 117,381 trips with 0 passengers, and trips with 7 to 9 passengers which is more than a taxi can carry.
- **Wrong amounts**: 7,127 trips with a negative total amount (probably refunds or corrections), and huge fares like $623,259.86 for 2.4 miles or $355,676.98 for 0 miles.
- **RatecodeID**: 252 trips have a code that is not between 1 and 6 (99), which are the only codes in the data dictionary.

These outliers change the averages and the min/max values a lot, so they should be filtered before a more serious analysis.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

#### Load the taxi zone lookup

In [33]:
# set dl url for the taxi zone lookup table
zone_lookup_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

# get the data
response = requests.get(zone_lookup_url)

# check that response was good and save the data
taxi_zone_lookup = "taxi_zone_lookup.csv"
if response.status_code == 200:
    with open(taxi_zone_lookup, "wb") as f:
        f.write(response.content)

In [18]:
# csv is not self describing, the file is small so we can infer the schema on all of it
df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("inferSchema", "true") \
    .load(taxi_zone_lookup)

df_zones.printSchema()
df_zones.show(5)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [19]:
# join the pickup and the dropoff location ids with the zone lookup to get the boroughs
pickup_zones = df_zones.select(F.col("LocationID").alias("PULocationID"), F.col("Borough").alias("pickup_borough"))
dropoff_zones = df_zones.select(F.col("LocationID").alias("DOLocationID"), F.col("Borough").alias("dropoff_borough"))

df_trips_boroughs = df_trips \
    .join(pickup_zones, on="PULocationID", how="left") \
    .join(dropoff_zones, on="DOLocationID", how="left")
df_jan_2019_boroughs = df_trips_boroughs.filter(
    (F.col("tpep_pickup_datetime") >= "2019-01-01") & (F.col("tpep_pickup_datetime") < "2019-02-01")
)

df_trips_boroughs.select("trip_id", "PULocationID", "pickup_borough", "DOLocationID", "dropoff_borough").show(5)

+-----------+------------+--------------+------------+---------------+
|    trip_id|PULocationID|pickup_borough|DOLocationID|dropoff_borough|
+-----------+------------+--------------+------------+---------------+
|60129542144|         151|     Manhattan|         239|      Manhattan|
|60129542145|         239|     Manhattan|         246|      Manhattan|
|60129542146|         236|     Manhattan|         236|      Manhattan|
|60129542147|         193|        Queens|         193|         Queens|
|60129542148|         193|        Queens|         193|         Queens|
+-----------+------------+--------------+------------+---------------+
only showing top 5 rows


#### which borough had most pickups? dropoffs?

In [20]:
df_trips_boroughs.groupBy("pickup_borough").count().orderBy(F.desc("count")).show()
df_trips_boroughs.groupBy("dropoff_borough").count().orderBy(F.desc("count")).show()

+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|6950965|
|        Queens| 471173|
|       Unknown| 159815|
|      Brooklyn|  91905|
|         Bronx|  18062|
|           N/A|   3890|
|           EWR|    446|
| Staten Island|    361|
+--------------+-------+

+---------------+-------+
|dropoff_borough|  count|
+---------------+-------+
|      Manhattan|6817355|
|         Queens| 340972|
|       Brooklyn| 301105|
|        Unknown| 149097|
|          Bronx|  58085|
|            N/A|  16904|
|            EWR|  10914|
|  Staten Island|   2185|
+---------------+-------+



- Most pickups: Manhattan by far with 6,950,965 pickups .
- Most dropoffs: Manhattan too with 6,817,355 dropoffs.

Queens is second (JFK and LaGuardia airports are in Queens). Brooklyn has a lot more dropoffs (301k) than pickups (92k)
Unknown (LocationID 264) and N/A (LocationID 265, outside of NYC) are not real boroughs.

#### what are the busy/slow times by borough

In [34]:
#number of trips by pickup borough and pickup hour
borough_hours = df_jan_2019_boroughs \
    .groupBy("pickup_borough", F.hour("tpep_pickup_datetime").alias("pickup_hour")) \
    .count()

# for each borough keep the hour with the most trips and the hour with the least trips
borough_hours.groupBy("pickup_borough") \
    .agg(

        
        F.max_by("pickup_hour", "count").alias("busiest_hour"),
        F.max("count").alias("trips_busiest_hour"),
        F.min_by("pickup_hour", "count").alias("slowest_hour"),
        
        F.min("count").alias("trips_slowest_hour")
    ) \
    .orderBy("pickup_borough") \
    .show()

+--------------+------------+------------------+------------+------------------+
|pickup_borough|busiest_hour|trips_busiest_hour|slowest_hour|trips_slowest_hour|
+--------------+------------+------------------+------------+------------------+
|         Bronx|           7|              1803|           2|               225|
|      Brooklyn|           8|              6935|           3|              1919|
|           EWR|          15|                54|           1|                 1|
|     Manhattan|          18|            471524|           4|             53446|
|           N/A|          19|               214|           6|                88|
|        Queens|          16|             29880|           3|              3084|
| Staten Island|           8|                36|           1|                 3|
|       Unknown|          18|             10751|           4|              1465|
+--------------+------------+------------------+------------+------------------+



- Manhattan and Unknown: busiest at 18h, slowest at 4h.
- Queens: busiest at 16h , slowest at 3h.
- Brooklyn, Bronx and Staten Island: busiest in the morning (7h-8h), slowest at night (1h-3h).
- EWR, N/A and Staten Island have very few trips so their results are not really meaningful.

For all the boroughs the slowest time is during the night. 

#### what are the busiest days of the week by borough?

In [22]:
# number of trips per day and borough, then average by day of the week (like in part 1)
borough_days = df_jan_2019_boroughs \
    .groupBy("pickup_borough", F.to_date("tpep_pickup_datetime").alias("pickup_date")) \
    .count() \
    .withColumn("day_of_week", F.date_format("pickup_date", "EEEE")) \
    .groupBy("pickup_borough", "day_of_week") \
    .agg(F.avg("count").alias("avg_trips_per_day"))

borough_days.groupBy("pickup_borough") \
    .agg(
        F.max_by("day_of_week", "avg_trips_per_day").alias("busiest_day"),
        F.max("avg_trips_per_day").alias("avg_trips_busiest_day")
    ) \
    .orderBy("pickup_borough") \
    .show()

+--------------+-----------+---------------------+
|pickup_borough|busiest_day|avg_trips_busiest_day|
+--------------+-----------+---------------------+
|         Bronx|     Friday|                666.5|
|      Brooklyn|     Friday|               3273.0|
|           EWR|     Friday|                 18.5|
|     Manhattan|     Friday|             246224.0|
|           N/A|    Tuesday|                140.6|
|        Queens|     Monday|              16662.0|
| Staten Island|     Friday|                 16.0|
|       Unknown|   Thursday|               5785.4|
+--------------+-----------+---------------------+



**Friday** is the busiest day of the week for most of the boroughs (Manhattan, Brooklyn, Bronx, Staten Island and EWR).  
**Queens** is busiest on **Monday** (probably the airports, with people coming back after the weekend). N/A is busiest on Tuesday and Unknown on Thursday.

#### what is the average trip distance by borough?

In [23]:
df_trips_boroughs.groupBy("pickup_borough") \
    .agg(F.avg("trip_distance").alias("avg_trip_distance")) \
    .orderBy(F.desc("avg_trip_distance")) \
    .show()

+--------------+------------------+
|pickup_borough| avg_trip_distance|
+--------------+------------------+
| Staten Island|12.503601108033246|
|        Queens|11.283218499361993|
|         Bronx| 7.233194552098303|
|      Brooklyn| 4.787677275447492|
|           N/A| 3.193850899742941|
|           EWR| 2.641098654708519|
|       Unknown| 2.415464130400774|
|     Manhattan|2.2286693358402596|
+--------------+------------------+



#### what is the average trip fare by borough?

In [24]:
df_trips_boroughs.groupBy("pickup_borough") \
    .agg(F.avg("fare_amount").alias("avg_fare_amount")) \
    .orderBy(F.desc("avg_fare_amount")) \
    .show()

+--------------+------------------+
|pickup_borough|   avg_fare_amount|
+--------------+------------------+
|           EWR| 76.24024663677126|
|           N/A|  59.5731593830335|
| Staten Island|45.289861495844896|
|        Queens| 35.14462651722029|
|         Bronx| 26.26890543682963|
|      Brooklyn|18.649132800172286|
|       Unknown|14.944423051653523|
|     Manhattan|10.792468572351568|
+--------------+------------------+



- Highest average fare: EWR  with 76.24dollars then N/A (outside of NYC) with 59.57 dollars, Staten Island (45.29) and Queens (35.14).
- Lowest average fare: Manhattan with 10.79 dollars.

The fares mostly follow the distances. EWR is strange: a high fare but a short average distance (2.6 miles).

#### highest/lowest faire amounts for a trip, what burough is associated with the each

In [25]:
fare_cols = ["trip_id", "fare_amount", "pickup_borough", "dropoff_borough", "trip_distance", "trip_duration_min"]

print("highest fare")
df_trips_boroughs.orderBy(F.desc("fare_amount")).select(fare_cols).show(1)
print("lowest fare")
df_trips_boroughs.orderBy(F.asc("fare_amount")).select(fare_cols).show(1)

highest fare
+-----------+-----------+--------------+---------------+-------------+-----------------+
|    trip_id|fare_amount|pickup_borough|dropoff_borough|trip_distance|trip_duration_min|
+-----------+-----------+--------------+---------------+-------------+-----------------+
|60132041799|  623259.86|     Manhattan|      Manhattan|          2.4|             19.9|
+-----------+-----------+--------------+---------------+-------------+-----------------+
only showing top 1 row
lowest fare
+-----------+-----------+--------------+---------------+-------------+-----------------+
|    trip_id|fare_amount|pickup_borough|dropoff_borough|trip_distance|trip_duration_min|
+-----------+-----------+--------------+---------------+-------------+-----------------+
|60134432793|     -362.0|        Queens|         Queens|          0.0|             6.85|
+-----------+-----------+--------------+---------------+-------------+-----------------+
only showing top 1 row


- Highest fare: 623,259.86 dollars, trip `60132041799`, from Manhattan to Manhattan (2.4 miles, 20 minutes), so it is an error.
- Lowest fare: -362.00 dollars, trip `60134432793`, from Queens to Queens (0 miles). A negative fare is probably a refund or a correction of a trip.

#### load the dataset from the most recently available january, is there a change to any of the average metrics

In [26]:
# the most recently available january is january 2026, the instructions at the top also ask for january 2025
# so we load both, using the same code as for january 2019
recent_trip_data = {}
for year in [2025, 2026]:
    download_url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-01.parquet'
    response = requests.get(download_url)
    file_name = f"yellow_tripdata_{year}-01.parquet"
    if response.status_code == 200:
        with open(file_name, "wb") as f:
            f.write(response.content)
    recent_trip_data[year] = file_name

df_trips_2025 = spark.read.parquet(recent_trip_data[2025])
df_trips_2026 = spark.read.parquet(recent_trip_data[2026])

In [27]:
def average_metrics(df, year):
    return df.withColumn(
        "trip_duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    ).agg(
        F.lit(year).alias("year"),
        F.count("*").alias("nb_trips"),
        F.avg("passenger_count").alias("avg_passenger_count"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.max("trip_distance").alias("max_trip_distance"),
        F.avg("trip_duration_min").alias("avg_trip_duration_min"),
        F.avg("fare_amount").alias("avg_fare_amount"),
        F.avg("tip_amount").alias("avg_tip_amount"),
        F.avg("total_amount").alias("avg_total_amount")
    )

average_metrics(df_trips, 2019) \
    .union(average_metrics(df_trips_2025, 2025)) \
    .union(average_metrics(df_trips_2026, 2026)) \
    .show()

+----+--------+-------------------+------------------+-----------------+---------------------+------------------+------------------+------------------+
|year|nb_trips|avg_passenger_count| avg_trip_distance|max_trip_distance|avg_trip_duration_min|   avg_fare_amount|    avg_tip_amount|  avg_total_amount|
+----+--------+-------------------+------------------+-----------------+---------------------+------------------+------------------+------------------+
|2019| 7696617| 1.5670317144945614|2.8301461681153532|            831.8|   16.551081570422276| 12.52967677747685|1.8208300763883147| 15.81065134371489|
|2025| 3475226| 1.2978589658806226| 5.855126178843539|        276423.57|    15.01811561799608| 17.08180276045484|2.9598127862758044|25.611291697280986|
|2026| 3724889|  1.256271258946819| 6.455646860885151|        269097.48|   17.193160337574085|20.804253893199448|2.6081422399430054|29.178525515798285|
+----+--------+-------------------+------------------+-----------------+----------------

In [28]:
# average trip distance and fare by borough for the 3 januaries
def borough_metrics(df, year):
    return df.join(pickup_zones, on="PULocationID", how="left") \
        .groupBy("pickup_borough") \
        .agg(
            F.avg("trip_distance").alias(f"avg_distance_{year}"),
            F.avg("fare_amount").alias(f"avg_fare_{year}")
        )

borough_metrics(df_trips, 2019) \
    .join(borough_metrics(df_trips_2025, 2025), on="pickup_borough", how="outer") \
    .join(borough_metrics(df_trips_2026, 2026), on="pickup_borough", how="outer") \
    .orderBy("pickup_borough") \
    .show()

+--------------+------------------+------------------+------------------+------------------+-------------------+------------------+
|pickup_borough| avg_distance_2019|     avg_fare_2019| avg_distance_2025|     avg_fare_2025|  avg_distance_2026|     avg_fare_2026|
+--------------+------------------+------------------+------------------+------------------+-------------------+------------------+
|         Bronx| 7.233194552098303| 26.26890543682963| 65.91340478936317|27.772212197272932| 12.283880876213214| 32.48834490050418|
|      Brooklyn| 4.787677275447492|18.649132800172286|24.807041925231022|23.436922809141773| 16.840026980496987|30.850515006292362|
|           EWR| 2.641098654708519| 76.24024663677126|0.8962864721485412| 81.47989389920424|0.47888888888888886| 89.72720164609055|
|     Manhattan|2.2286693358402596|10.792468572351568| 4.443918890354086|13.869130184266965|   5.09188978078639| 17.48394860791505|
|           N/A| 3.193850899742941|  59.5731593830335|28.235833333333332| 78

The most recent January available is January 2026. We also loaded January 2025 because the instructions at the top of the notebook ask for it.

- Number of trips: 7.7M in 2019, only 3.5M in 2025 and 3.7M in 2026, less than half (competition with Uber/Lyft).
- Passenger count: goes down from 1.57 to 1.30 and 1.26.
- Fare: goes up from 12.53 dollars to 17.08 d and 20.80 d, and the total amount from 15.81 d to 25.61 d and 29.18 d (fares went up and there are new fees, like `cbd_congestion_fee` that exists since 2025).
- Tip: goes up from 1.82d to 2.96 d (2025) and 2.61 d (2026).
- Duration: almost the same (15-17 minutes).
  
- Distance: the average goes from 2.83 to 5.86 and 6.46 miles, but it's not a real change: the max distance is about 276,000 miles in 2025 and 269,000 in 2026.

By borough, the average fare went up almost everywhere (Manhattan 10.79 to 13.87 to 17.48).

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

#### Part 3 answers

The main work was done with pyspark, so here we redo 3 questions in pure SQL:
1. What is the average passenger count
2. Busiest day/slowest single day
3. Which borough had most pickups? dropoffs? (with a join on the zone lookup table)

In [29]:
#register the dataframes as tables to query them with sql
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

#### What is the Average passenger count (sql)

In [30]:
spark.sql("""
    SELECT AVG(passenger_count) AS avg_passenger_count
    FROM trips
""").show()

+-------------------+
|avg_passenger_count|
+-------------------+
| 1.5670317144945614|
+-------------------+



Same result as with pyspark: about **1.57 passengers** per trip.

#### busiest day/slowest single day (sql)

In [31]:
spark.sql("""
    WITH daily_trips AS (
        SELECT TO_DATE(tpep_pickup_datetime) AS pickup_date, COUNT(*) AS count
        FROM trips
        WHERE tpep_pickup_datetime >= '2019-01-01' AND tpep_pickup_datetime < '2019-02-01'
        GROUP BY TO_DATE(tpep_pickup_datetime)
    )
    (SELECT 'busiest' AS type, pickup_date, count FROM daily_trips ORDER BY count DESC LIMIT 1)
    UNION ALL
    (SELECT 'slowest' AS type, pickup_date, count FROM daily_trips ORDER BY count ASC LIMIT 1)
""").show()

+-------+-----------+------+
|   type|pickup_date| count|
+-------+-----------+------+
|busiest| 2019-01-25|292499|
|slowest| 2019-01-01|189432|
+-------+-----------+------+



Same result as with pyspark: the busiest day is 2019-01-25 with 292,499 trips and the slowest is 2019-01-01 with 189,432 trips

#### which borough had most pickups? dropoffs? (sql, with a join)

In [32]:
spark.sql("""
    SELECT z.Borough AS pickup_borough, COUNT(*) AS count
    FROM trips t
    LEFT JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY count DESC
""").show()

spark.sql("""
    SELECT z.Borough AS dropoff_borough, COUNT(*) AS count
    FROM trips t
    LEFT JOIN zones z ON t.DOLocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY count DESC
""").show()

+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|6950965|
|        Queens| 471173|
|       Unknown| 159815|
|      Brooklyn|  91905|
|         Bronx|  18062|
|           N/A|   3890|
|           EWR|    446|
| Staten Island|    361|
+--------------+-------+

+---------------+-------+
|dropoff_borough|  count|
+---------------+-------+
|      Manhattan|6817355|
|         Queens| 340972|
|       Brooklyn| 301105|
|        Unknown| 149097|
|          Bronx|  58085|
|            N/A|  16904|
|            EWR|  10914|
|  Staten Island|   2185|
+---------------+-------+



Same result as with pyspark: Manhattan has the most pickups (6,950,965) and the most dropoffs (6,817,355).

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing